<a href="https://www.kaggle.com/code/mohitrajemdramahajan/ats-based-resume-scorer-or-recommender?scriptVersionId=321957205" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Project Title: ATS BASED RESUME SCORER AND RECOMMENDER

In [1]:
!pip install gradio pdfplumber python-docx nltk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 82.7 MB/s eta 0:00:00


In [2]:
import re
from pathlib import Path
import gradio as gr
import pdfplumber
import docx
import nltk

display("All required libraries are imported sucessfully!")

'All required libraries are imported sucessfully!'

# Download NLTK Tokenizer Resources

In [3]:
nltk.download("punkt", quiet=True)

try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

display("The NLTK file is downloaded sucessfully! ")

'The NLTK file is downloaded sucessfully! '

# Text Extraction from PDF Files

In [4]:
# function for the text extraction from the pdf
def extract_text_from_pdf(file_path):
    """
    Extracts text from a PDF resume using pdfplumber.
    
    Parameters:
        file_path: Path of the uploaded PDF file.
        
    Returns:
        A string containing all extracted text from the PDF.
    """
    text = ""

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"
    return text.strip()

In [5]:
# function for the text extraction from the docx file
def extract_text_from_docx(file_path):
    """
    Extracts text from a DOCX resume using python-docx.
    
    Parameters:
        file_path: Path of the uploaded DOCX file.
        
    Returns:
        A string containing all extracted text from the DOCX file.
    """
    document = docx.Document(file_path)
    text = ""

    for paragraph in document.paragraphs:
        if paragraph.text.strip():
            text += paragraph.text + "\n"

    return text.strip()


# Define ATS Evaluation Criteria

In [6]:
IMPORTANT_SECTIONS = [
    "education",
    "experience",
    "skills",
    "certifications",
    "projects"
]

IMPORTANT_KEYWORDS = [
    "teamwork",
    "communication",
    "leadership",
    "python",
    "machine learning",
    "project management"
]

# Resume Analysis Function

In [7]:
def analyze_resume(text):
    """
    Analyzes resume text and generates an ATS-style score with recommendations.
    
    Parameters:
        text: Extracted resume text.
        
    Returns:
        final_score: Resume score between 0 and 100.
        recommendations: List of improvement suggestions.
    """
    score = 0
    recommendations = []

    text_lower = text.lower()

    # 1. Section presence check
    found_sections = [
        section for section in IMPORTANT_SECTIONS
        if section in text_lower
    ]

    score += len(found_sections) * 15

    missing_sections = list(set(IMPORTANT_SECTIONS) - set(found_sections))

    if missing_sections:
        recommendations.append(
            "Missing important sections: "
            + ", ".join(missing_sections).title()
            + "."
        )

    # 2. Resume length check
    words = nltk.word_tokenize(text)
    word_count = len(words)

    if word_count < 150:
        score -= 10
        recommendations.append(
            "Resume is too short. Ideal length is approximately 400 to 600 words."
        )
    elif word_count > 800:
        score -= 10
        recommendations.append(
            "Resume is too long. Try keeping it concise and preferably under two pages."
        )

    # 3. Keyword check
    found_keywords = [
        keyword for keyword in IMPORTANT_KEYWORDS
        if keyword in text_lower
    ]

    score += len(found_keywords) * 5

    if len(found_keywords) < 3:
        recommendations.append(
            "Consider adding relevant keywords such as teamwork, leadership, Python, "
            "machine learning, or project management where appropriate."
        )

    # 4. Date/year check
    if not re.search(r"\b\d{4}\b", text):
        score -= 5
        recommendations.append(
            "Add years to education and experience entries, such as 2022 or 2023."
        )

    # 5. Formatting and spacing check
    if text.count("\n") < 10:
        score -= 5
        recommendations.append(
            "Improve spacing and section formatting for better readability."
        )

    final_score = max(0, min(100, score))

    return final_score, recommendations, word_count, found_sections, found_keywords

# Process Uploaded Resume

In [8]:
def process_resume(file):
    """
    Processes an uploaded resume file and returns ATS analysis results.
    
    Parameters:
        file: Uploaded resume file from Gradio.
        
    Returns:
        Markdown-formatted analysis result.
    """
    if file is None:
        return "Please upload a PDF or DOCX resume."

    file_path = Path(file.name)
    file_extension = file_path.suffix.lower()

    if file_extension == ".pdf":
        text = extract_text_from_pdf(file_path)
    elif file_extension == ".docx":
        text = extract_text_from_docx(file_path)
    else:
        return "Unsupported file format. Please upload a PDF or DOCX file."

    if not text:
        return "No readable text was found in the uploaded resume."

    score, recommendations, word_count, found_sections, found_keywords = analyze_resume(text)

    output = f"""
# ATS Resume Analysis Result

## Resume Score: {score}%

**Word Count:** {word_count}

**Detected Sections:** {", ".join(found_sections).title() if found_sections else "None"}

**Detected Keywords:** {", ".join(found_keywords).title() if found_keywords else "None"}

## Recommendations
"""

    if recommendations:
        for recommendation in recommendations:
            output += f"- {recommendation}\n"
    else:
        output += "- Your resume satisfies the basic ATS evaluation criteria.\n"

    return output

# Create Gradio Interface

In [9]:
demo = gr.Interface(
    fn=process_resume,
    inputs=gr.File(
        label="Upload Resume",
        file_types=[".pdf", ".docx"]
    ),
    outputs=gr.Markdown(label="ATS Analysis Result"),
    title="ATS-Based Resume Scorer and Recommender",
    description=(
        "Upload a resume in PDF or DOCX format. "
        "The system evaluates the resume using rule-based ATS criteria "
        "and provides a score with improvement recommendations."
    )
)

# Launch the Application

In [10]:
# Create a dummy file object to simulate Gradio file upload
class DummyFile:
    def __init__(self, name):
        self.name = name

# IMPORTANT: Replace 'path/to/your/sample_resume.pdf' with the actual path to a PDF or DOCX resume file
# For example, you might upload a sample resume to your Colab environment and use its path here.
sample_resume_path = "/kaggle/input/datasets/mohitrajemdramahajan/sample-resume/MRMRUM1.pdf" # <--- UPDATE THIS PATH

# Create a dummy file object for process_resume
dummy_file_obj = DummyFile(sample_resume_path)

# Process the dummy resume and display the result
analysis_result = process_resume(dummy_file_obj)
print(analysis_result)


# ATS Resume Analysis Result

## Resume Score: 70%

**Word Count:** 560

**Detected Sections:** Education, Experience, Skills, Projects

**Detected Keywords:** Python, Machine Learning

## Recommendations
- Missing important sections: Certifications.
- Consider adding relevant keywords such as teamwork, leadership, Python, machine learning, or project management where appropriate.



In [11]:
# Create a dummy file object to simulate Gradio file upload
class DummyFile:
    def __init__(self, name):
        self.name = name

# IMPORTANT: Replace 'path/to/your/sample_resume.pdf' with the actual path to a PDF or DOCX resume file
# For example, you might upload a sample resume to your Colab environment and use its path here.
sample_resume_path = "/kaggle/input/datasets/mohitrajemdramahajan/sample-resume-2/06-05-2026 interview game dev.pdf" # <--- UPDATE THIS PATH

# Create a dummy file object for process_resume
dummy_file_obj = DummyFile(sample_resume_path)

# Process the dummy resume and display the result
analysis_result_1 = process_resume(dummy_file_obj)
print("Processed Sucessfully")
print(analysis_result_1)

Processed Sucessfully

# ATS Resume Analysis Result

## Resume Score: 15%

**Word Count:** 246

**Detected Sections:** Experience

**Detected Keywords:** Machine Learning

## Recommendations
- Missing important sections: Certifications, Skills, Education, Projects.
- Consider adding relevant keywords such as teamwork, leadership, Python, machine learning, or project management where appropriate.
- Add years to education and experience entries, such as 2022 or 2023.

